In [1]:
import pandas as pd
import yfinance as yf
import FinanceDataReader as fdr
from datetime import datetime

# ==========================================
# 1. 가격 데이터 추출 (실시간 주가 및 환율)
# ==========================================
def fetch_current_prices():
    # 달러 환율 가져오기
    exchange_rate = yf.Ticker("USDKRW=X").history(period="1d")['Close'].iloc[-1]

    # 각 자산별 최신 종가 세팅
    prices = {
        'BTC': yf.Ticker("BTC-USD").history(period="1d")['Close'].iloc[-1] * exchange_rate,
        'ETH': yf.Ticker("ETH-USD").history(period="1d")['Close'].iloc[-1] * exchange_rate,
        'QQQ': yf.Ticker("QQQ").history(period="1d")['Close'].iloc[-1] * exchange_rate,
        'GOOGL': yf.Ticker("GOOGL").history(period="1d")['Close'].iloc[-1] * exchange_rate,
        'CRCL': yf.Ticker("CRCL").history(period="1d")['Close'].iloc[-1] * exchange_rate,
        'SK_Hynix': fdr.DataReader('000660').iloc[-1]['Close'],  # 국내 자산
        'USD': exchange_rate,
        'KRW': 1.0
    }
    return prices, exchange_rate

# ==========================================
# 2. 데이터 변환 및 우종님 전용 상대적 밴드(±20%) 룰 적용
# ==========================================
def process_portfolio_data(input_csv, output_csv):
    df = pd.read_csv(input_csv)
    prices, exchange_rate = fetch_current_prices()

    # 현재 가치 및 비중 계산
    df['Current_Price'] = df['Asset'].map(prices)
    df['Total_Value_KRW'] = df['Quantity'] * df['Current_Price']

    total_asset_value = df['Total_Value_KRW'].sum()
    df['Current_Weight_%'] = (df['Total_Value_KRW'] / total_asset_value) * 100

    # 목표 비중과의 격차 (양수: 부족, 음수: 초과)
    df['Weight_Gap_%'] = df['Target_Weight'] - df['Current_Weight_%']

    actions = []
    action_amounts_krw = []
    action_amounts_usd = []
    action_quantities = []

    usd_assets = ['QQQ', 'GOOGL', 'CRCL']

    for _, row in df.iterrows():
        asset = row['Asset']
        target_w = row['Target_Weight']
        current_w = row['Current_Weight_%']
        current_price = row['Current_Price']

        # 목표 비중으로 되돌리기 위해 필요한 원화 금액
        required_krw = total_asset_value * (row['Weight_Gap_%'] / 100)

        # [우종님의 매매 룰 엔진: 상대적 ±20% 밴드]
        if asset in ['KRW', 'USD']:
            # 현금은 매매 대상이 아닌 조절 수단이므로 신호에서 제외
            exec_amount_krw = 0
            actions.append("CASH_BUFFER")
        else:
            buy_threshold = target_w * 0.8  # 20% 부족할 때 (하단 밴드)
            sell_threshold = target_w * 1.2 # 20% 초과할 때 (상단 밴드)

            if current_w <= buy_threshold:
                # 하단 이탈: 목표 비중까지 전액 매수
                exec_amount_krw = required_krw
                actions.append("BUY (상대적 저점 매수)")
            elif current_w >= sell_threshold:
                # 상단 이탈: 목표 비중까지 초과분 매도
                exec_amount_krw = abs(required_krw)
                actions.append("SELL (상대적 고점 익절)")
            else:
                # 밴드 내부: 관망 (정상 범위)
                exec_amount_krw = 0
                actions.append("HOLD (안전지대)")

        action_amounts_krw.append(exec_amount_krw)

        # 미국 주식 달러 환산
        if asset in usd_assets and exec_amount_krw > 0:
            action_amounts_usd.append(exec_amount_krw / exchange_rate)
        else:
            action_amounts_usd.append(0)

        # 권장 수량 계산 (한국 주식은 정수, 나머지는 소수점 유지)
        if current_price > 1.0 and exec_amount_krw > 0 and asset not in ['KRW', 'USD']:
            raw_qty = exec_amount_krw / current_price
            if asset == 'SK_Hynix':
                final_quantity = round(raw_qty)
            else:
                final_quantity = raw_qty
            action_quantities.append(final_quantity)
        else:
            action_quantities.append(0)

    # 테블로 연동용 데이터 컬럼 생성
    df['Action_Signal'] = actions
    df['Action_Amount_KRW'] = action_amounts_krw
    df['Action_Amount_USD'] = action_amounts_usd
    df['Action_Quantity'] = action_quantities
    df['Last_Updated'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    # ==========================================
    # 3. 테블로 연동 파일 저장 및 결과 출력
    # ==========================================
    df.to_csv(output_csv, index=False)

    print(f"\n🚀 [최종 퀀트 엔진] 상대적 리밸런싱 지시서 생성 완료 (±20% 밴드)")
    print(f"💰 총 자산 가치: {total_asset_value:,.0f} 원")
    print("-" * 115)
    print(f"{'Asset':<10} | {'목표':>5}% | {'현재':>5}% | {'상태':<20} | {'주문금액(통화맞춤)':>15} | {'권장수량'}")
    print("-" * 115)

    for _, row in df.iterrows():
        asset = row['Asset']
        amt_krw, amt_usd = row['Action_Amount_KRW'], row['Action_Amount_USD']
        qty = row['Action_Quantity']

        # 통화 맞춤형 텍스트 포맷팅
        if asset in usd_assets and amt_usd > 0:
            amt_str = f"${amt_usd:,.2f}"
        elif amt_krw > 0:
            amt_str = f"{amt_krw:,.0f}원"
        else:
            amt_str = "-"

        # 수량 맞춤형 텍스트 포맷팅 (정수 vs 소수점)
        if asset == 'SK_Hynix' and qty > 0:
            qty_str = f"{int(qty)} 주"
        elif qty > 0 and asset not in ['KRW', 'USD']:
            qty_str = f"{qty:.4f} 개"
        else:
            qty_str = "-"

        print(f"{asset:<10} | {row['Target_Weight']:>6.1f} | {row['Current_Weight_%']:>6.1f} | {row['Action_Signal']:<20} | {amt_str:>16} | {qty_str}")

# 매달 8일 혹은 필요시 실행
process_portfolio_data('my_assets.csv', 'portfolio_dashboard_data.csv')


🚀 [최종 퀀트 엔진] 상대적 리밸런싱 지시서 생성 완료 (±20% 밴드)
💰 총 자산 가치: 30,326,450 원
-------------------------------------------------------------------------------------------------------------------
Asset      |    목표% |    현재% | 상태                   |      주문금액(통화맞춤) | 권장수량
-------------------------------------------------------------------------------------------------------------------
BTC        |   50.0 |   49.9 | HOLD (안전지대)          |                - | -
ETH        |   10.5 |   10.7 | HOLD (안전지대)          |                - | -
QQQ        |    9.6 |    9.7 | HOLD (안전지대)          |                - | -
SK_Hynix   |    4.8 |    3.4 | BUY (상대적 저점 매수)      |         437,833원 | -
GOOGL      |    4.8 |    4.9 | HOLD (안전지대)          |                - | -
KRW        |    3.0 |    3.4 | CASH_BUFFER          |                - | -
USD        |    7.0 |    7.9 | CASH_BUFFER          |                - | -
CRCL       |   10.2 |   10.1 | HOLD (안전지대)          |                - | -
